# Generalized Advantage Estimation visualization

This notebook visualizes how the GAE `lambda` parameter interpolates between a one-step TD(0) target and a full Monte Carlo return.

For an episode with rewards `r_t`, value estimates `V(s_t)`, and terminal `V(s_T) = 0`:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

$$A_t^{GAE(\lambda)} = \sum_{l=0}^{T-t-1} (\gamma \lambda)^l \delta_{t+l}$$

$$R_t^{GAE(\lambda)} = V(s_t) + A_t^{GAE(\lambda)}$$

The return target satisfies:

- `lambda = 0`: `R_GAE` is the TD(0) target `r_t + gamma V(s_{t+1})`.
- `lambda = 1`: `R_GAE` is the Monte Carlo return for a finite episode.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

plt.rcParams.update(
    {
        "figure.figsize": (12, 4),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "legend.frameon": False,
    }
)

## Sparse-reward episode

The episode below has mostly zero rewards, a small early positive reward, a delayed penalty, and a large terminal reward. The value function is deliberately imperfect so the TD(0) target and Monte Carlo return look meaningfully different.

In [ ]:
gamma = 0.99
T = 300
steps = np.arange(T)

rewards = np.zeros(T)
reward_steps = np.array([6, 17, 29])*5
rewards[reward_steps] = np.array([1.0, -0.4, 2.5])

# One value estimate per state s_0 ... s_T. The final state is terminal.
state_steps = np.arange(T + 1)
values = (
    0.18 * np.sin(np.linspace(0.0, 3.0 * np.pi, T + 1))
    + 0.25 * np.exp(-0.5 * ((state_steps - 6*5) / 2.0) ** 2)
    + 0.85 * np.exp(-0.5 * ((state_steps - 29*5) / 5.0) ** 2)
)
values[-1] = 0.0
values *= 2.0

fig, axes = plt.subplots(
    2,
    1,
    sharex=True,
    figsize=(12, 5),
    gridspec_kw={"height_ratios": [1, 2]},
)

markerline, stemlines, baseline = axes[0].stem(steps, rewards, basefmt=" ")
plt.setp(markerline, marker="o", markersize=6, color="tab:green")
plt.setp(stemlines, linewidth=2, color="tab:green")
axes[0].axhline(0.0, color="black", linewidth=0.8, alpha=0.5)
axes[0].set_ylabel("reward")
axes[0].set_title("Sparse rewards")

axes[1].plot(state_steps, values, marker="o", markersize=3, color="tab:blue", label="critic V(s_t)")
axes[1].scatter([T], [values[-1]], color="tab:red", zorder=3, label="terminal V(s_T) = 0")
axes[1].set_xlabel("time step")
axes[1].set_ylabel("value estimate")
axes[1].set_title("Imperfect value estimates used for bootstrapping")
axes[1].legend(loc="upper left")

plt.tight_layout()

## Return calculations

`td0_targets` bootstraps after one step. `monte_carlo_returns` waits until the end of the episode. `gae_returns` adds the GAE advantage back to the current value estimate so all three curves are comparable as value targets.

In [ ]:
def monte_carlo_returns(rewards, gamma):
    returns = np.zeros_like(rewards, dtype=float)
    running_return = 0.0

    for t in reversed(range(len(rewards))):
        running_return = rewards[t] + gamma * running_return
        returns[t] = running_return

    return returns


def td0_targets(rewards, values, gamma):
    return rewards + gamma * values[1:]


def td_errors(rewards, values, gamma):
    return rewards + gamma * values[1:] - values[:-1]


def gae_advantages(rewards, values, gamma, lam):
    deltas     = td_errors(rewards, values, gamma)
    advantages = np.zeros_like(rewards, dtype=float)
    running_advantage = 0.0

    for t in reversed(range(len(rewards))):
        running_advantage = deltas[t] + gamma * lam * running_advantage
        advantages[t] = running_advantage

    return advantages


def gae_returns(rewards, values, gamma, lam):
    return values[:-1] + gae_advantages(rewards, values, gamma, lam)


mc = monte_carlo_returns(rewards, gamma)
td0 = td0_targets(rewards, values, gamma)
deltas = td_errors(rewards, values, gamma)

lambdas = [0.0, 0.25, 0.5, 0.75, 0.95, 0.99, 1.0]
gae_by_lambda = {lam: gae_returns(rewards, values, gamma, lam) for lam in lambdas}

print("max |GAE return(lambda=0) - TD0 target|:", np.max(np.abs(gae_by_lambda[0.0] - td0)))
print("max |GAE return(lambda=1) - MC return|:  ", np.max(np.abs(gae_by_lambda[1.0] - mc)))

## TD residuals

GAE is a discounted sum of TD residuals. In sparse-reward problems, most rewards are zero, but the residuals can still be nonzero because the critic changes from one state to the next.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.5)
ax.bar(steps, deltas, width=0.85, color="tab:purple", alpha=0.75, label="TD residual delta_t")
for reward_step in reward_steps:
    ax.axvline(reward_step, color="tab:green", linewidth=1.5, linestyle=":", alpha=0.8)
ax.set_title("TD residuals used by GAE")
ax.set_xlabel("time step")
ax.set_ylabel("delta_t")
ax.legend(loc="upper left")
plt.tight_layout()

## GAE return between TD(0) and Monte Carlo

Low `lambda` stays close to the one-step TD target. High `lambda` carries delayed rewards farther backward and approaches the Monte Carlo return.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(steps, mc, color="black", linewidth=3, label="Monte Carlo return")
ax.plot(steps, td0, color="tab:red", linewidth=2.5, linestyle="--", label="TD(0) target")

for lam in [0.25, 0.5, 0.75, 0.95, 0.99]:
    ax.plot(
        steps,
        gae_by_lambda[lam],
        color=plt.cm.viridis(lam),
        linewidth=2,
        label=f"GAE return lambda={lam:g}",
    )

for reward_step in reward_steps:
    ax.axvline(reward_step, color="tab:green", linewidth=1.5, linestyle=":", alpha=0.8)

ax.set_title("GAE return interpolates from TD(0) to Monte Carlo")
ax.set_xlabel("time step")
ax.set_ylabel("target return")
ax.legend(loc="upper left", ncols=2)
plt.tight_layout()

## Full lambda sweep

Each row is one `lambda` value. Moving upward in the heatmap increases the effective horizon of the target.

In [ ]:
lambda_grid = np.linspace(0.0, 1.0, 101)
gae_grid = np.vstack([gae_returns(rewards, values, gamma, lam) for lam in lambda_grid])

fig, ax = plt.subplots(figsize=(13, 5))
limit = np.max(np.abs(gae_grid))
im = ax.imshow(
    gae_grid,
    aspect="auto",
    origin="lower",
    extent=[0, T - 1, 0, 1],
    cmap="coolwarm",
    vmin=-limit,
    vmax=limit,
)
for reward_step in reward_steps:
    ax.axvline(reward_step, color="black", linewidth=1.0, linestyle=":", alpha=0.6)

ax.set_title("GAE return target across lambda")
ax.set_xlabel("time step")
ax.set_ylabel("lambda")
fig.colorbar(im, ax=ax, label="GAE return target")
plt.tight_layout()

## Why lambda changes the horizon

The TD residual at lag `l` receives weight `(gamma * lambda)^l`. With `lambda = 0`, only the first residual contributes. As `lambda` approaches `1`, more future residuals survive, which makes the target more Monte-Carlo-like.

In [ ]:
lags = np.arange(T)

fig, ax = plt.subplots(figsize=(12, 4))
for lam in lambdas:
    ax.plot(lags, (gamma * lam) ** lags, linewidth=2, label=f"lambda={lam:g}")

ax.set_title("Weight on future TD residuals")
ax.set_xlabel("lag l")
ax.set_ylabel("weight (gamma * lambda)^l")
ax.legend(loc="upper right", ncols=2)
plt.tight_layout()

## Single-lambda view

Change `selected_lambda` and rerun this cell to inspect one GAE target against its two endpoints.

In [ ]:
selected_lambda = 0.9

selected_gae = gae_returns(rewards, values, gamma, selected_lambda)
selected_advantage = gae_advantages(rewards, values, gamma, selected_lambda)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

axes[0].plot(steps, mc, color="black", linewidth=2.5, label="Monte Carlo")
axes[0].plot(steps, td0, color="tab:red", linewidth=2.0, linestyle="--", label="TD(0)")
axes[0].plot(steps, selected_gae, color="tab:blue", linewidth=2.5, label=f"GAE return lambda={selected_lambda:g}")
axes[0].set_ylabel("target return")
axes[0].set_title("Selected GAE return target")
axes[0].legend(loc="upper left")

axes[1].axhline(0.0, color="black", linewidth=0.8, alpha=0.5)
axes[1].bar(steps, selected_advantage, width=0.85, color="tab:blue", alpha=0.75)
axes[1].set_xlabel("time step")
axes[1].set_ylabel("advantage")
axes[1].set_title("GAE advantage added to V(s_t)")

for ax in axes:
    for reward_step in reward_steps:
        ax.axvline(reward_step, color="tab:green", linewidth=1.5, linestyle=":", alpha=0.8)

plt.tight_layout()